# Tutorial 2: Structural Equation Modeling, Mediation & GPU BCa Bootstrap

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FredPedrosa/anvala/blob/main/notebooks/02_sem_mediation_gpu_bootstrap.ipynb)

This tutorial covers:
- Specifying full latent SEM models (`=~`, `~`, `~~`) in `lavaan` syntax
- Labeling paths (`a*X`, `b*M`) and defining nonlinear functions (`indirect := a*b`, `total := c + a*b`)
- Comparing analytical Delta-method standard errors vs. GPU-accelerated `BCa` and `percentile` bootstrap confidence intervals (`bootstrap_ci`)
- Applying parameter equality constraints by reusing parameter labels


In [1]:
import importlib.util
if importlib.util.find_spec("anvala") is None:
    get_ipython().run_line_magic("pip", "install -q anvala")

import numpy as np
import pandas as pd
import torch
import anvala as av

pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 100)
print(f"anvala loaded | PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")


anvala loaded | PyTorch 2.6.0+cu124 | CUDA available: True


## 1. Observed-Variable Mediation & GPU BCa Bootstrap


In [2]:
rng = np.random.default_rng(2026)
n = 350

X = rng.normal(0.0, 1.0, size=n)
M = 0.50 * X + rng.normal(0.0, 0.85, size=n)
Y = 0.45 * M + 0.18 * X + rng.normal(0.0, 0.85, size=n)
df_med = pd.DataFrame({"X": X, "M": M, "Y": Y})

med_syntax = """
    M ~ a*X
    Y ~ b*M + c*X
    indirect := a*b
    total    := c + a*b
    prop_med := (a*b) / (c + a*b)
"""

fit_med = av.sem(med_syntax, data=df_med, estimator="ML")
print("=== Analytical Estimates (Delta Method for :=) ===")
print(fit_med.parameter_estimates()[["lhs", "op", "rhs", "label", "est", "se", "pvalue", "std_all"]].round(4).to_string(index=False))

# GPU-accelerated BCa Bootstrap
boot_bca = fit_med.bootstrap_ci(n_boot=1000, conf_level=0.95, method="bca", seed=42)
print("\n=== BCa Bootstrap 95% CIs ===")
print(boot_bca.summary())

# Percentile Bootstrap for comparison
boot_pct = fit_med.bootstrap_ci(n_boot=1000, conf_level=0.95, method="percentile", seed=42)
print("\nIndirect effect CI comparison:")
print("  BCa       :", boot_bca.table.query("lhs == 'indirect'")[["est", "boot_se", "ci_lower", "ci_upper"]].round(4).to_dict("records")[0])
print("  Percentile:", boot_pct.table.query("lhs == 'indirect'")[["est", "boot_se", "ci_lower", "ci_upper"]].round(4).to_dict("records")[0])


=== Analytical Estimates (Delta Method for :=) ===
     lhs op               rhs    label    est     se  pvalue  std_all
       M  ~                 X        a 0.4983 0.0446  0.0000   0.5129
       Y  ~                 M        b 0.3789 0.0525  0.0000   0.3824
       Y  ~                 X        c 0.2047 0.0510  0.0001   0.2127
       X ~~                 X          1.0915 0.0000     NaN   1.0000
       M ~~                 M          0.7590 0.0574  0.0000   0.7369
       Y ~~                 Y          0.7330 0.0554  0.0000   0.7251
indirect :=               a*b indirect 0.1888 0.0312  0.0000   0.1962
   total :=           c + a*b    total 0.3935 0.0469  0.0000   0.4088
prop_med := (a*b) / (c + a*b) prop_med 0.4798 0.0882  0.0000   0.4798

=== BCa Bootstrap 95% CIs ===
anvala Nonparametric Bootstrap (B=1000, 95% BCA, device=cuda)
     lhs op               rhs    label    est  boot_se    bias  ci_lower  ci_upper  pvalue
       M  ~                 X        a 0.4983   0.0441 -0.0014   

## 2. Full Latent-Variable Structural Mediation Model


In [3]:
rng2 = np.random.default_rng(99)
n2 = 400

F_X = rng2.normal(0.0, 1.0, size=n2)
F_M = 0.55 * F_X + rng2.normal(0.0, 0.8, size=n2)
F_Y = 0.50 * F_M + 0.20 * F_X + rng2.normal(0.0, 0.8, size=n2)

def make_indicators(factor, loadings, prefix):
    out = {}
    for i, l in enumerate(loadings, start=1):
        out[f"{prefix}{i}"] = l * factor + rng2.normal(0.0, 0.55, size=len(factor))
    return out

df_lsem = pd.DataFrame({
    **make_indicators(F_X, [0.80, 0.78, 0.74], "x"),
    **make_indicators(F_M, [0.82, 0.76, 0.72], "m"),
    **make_indicators(F_Y, [0.85, 0.79, 0.73], "y"),
})

latent_sem_syntax = """
    # Measurement model
    FX =~ x1 + x2 + x3
    FM =~ m1 + m2 + m3
    FY =~ y1 + y2 + y3

    # Structural model
    FM ~ a*FX
    FY ~ b*FM + c*FX

    # Defined parameters
    indirect := a*b
    total    := c + a*b
"""

fit_lsem = av.sem(latent_sem_syntax, data=df_lsem, estimator="MLR")
print("=== Latent SEM Fit Measures (MLR) ===")
print(fit_lsem.fit_measures().round(4))

print("\n=== Structural Paths & Defined Parameters ===")
pe_lsem = fit_lsem.parameter_estimates()
print(pe_lsem[pe_lsem["op"].isin(["~", ":="])][["lhs", "op", "rhs", "label", "est", "se", "pvalue", "std_all"]].round(4).to_string(index=False))

# GPU BCa bootstrap for the full latent mediation model
boot_lsem = fit_lsem.bootstrap_ci(n_boot=500, conf_level=0.95, method="bca", seed=42)
print("\n=== Latent SEM BCa Bootstrap (Structural & Indirect Paths) ===")
print(boot_lsem.table[boot_lsem.table["op"].isin(["~", ":="])].round(4).to_string(index=False))


=== Latent SEM Fit Measures (MLR) ===
                Unscaled   Scaled  Robust
chisq            21.2029  21.8155     NaN
df               24.0000  24.0000     NaN
pvalue            0.6267   0.5903     NaN
scaling_factor    1.0000   0.9719     NaN
cfi               1.0000   1.0000  1.0000
tli               1.0022   1.0017  1.0017
rmsea             0.0000   0.0000  0.0000
srmr              0.0224   0.0224  0.0224

=== Structural Paths & Defined Parameters ===
     lhs op     rhs    label    est     se  pvalue  std_all
      FM  ~      FX        a 0.5848 0.0545  0.0000   0.6035
      FY  ~      FM        b 0.4695 0.0682  0.0000   0.4552
      FY  ~      FX        c 0.2451 0.0715  0.0006   0.2453
indirect :=     a*b indirect 0.2746 0.0474  0.0000   0.2747
   total := c + a*b    total 0.5197 0.0614  0.0000   0.5200

=== Latent SEM BCa Bootstrap (Structural & Indirect Paths) ===
     lhs op     rhs    label    est  boot_se    bias  ci_lower  ci_upper  pvalue
      FM  ~      FX        a 0.5

## 3. Equality Constraints & Nested Model Comparison


In [4]:
# Test whether a == b using shared parameter label 'eq_ab'
constrained_syntax = """
    FX =~ x1 + x2 + x3
    FM =~ m1 + m2 + m3
    FY =~ y1 + y2 + y3

    FM ~ eq_ab*FX
    FY ~ eq_ab*FM + c*FX
"""
fit_lsem_eq = av.sem(constrained_syntax, data=df_lsem, estimator="MLR")
print(av.compare_models(fit_lsem, fit_lsem_eq, names=["Free (a, b)", "Constrained (a = b)"])[["model", "df", "chisq_scaled", "chisq_diff", "df_diff", "pvalue", "cfi", "rmsea"]].round(4).to_string(index=False))


              model  df  chisq_scaled  chisq_diff  df_diff  pvalue  cfi  rmsea
        Free (a, b)  24       21.8155         NaN      NaN     NaN  1.0    0.0
Constrained (a = b)  25       23.4816      1.7855      1.0  0.1815  1.0    0.0
